# Librerias

In [1]:
import hashlib
import json
import os
import re
import sys
import time
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher
from zoneinfo import ZoneInfo

# Librerías de terceros
import geopandas as gpd
import numpy as np
import pandas as pd
import pyproj
import requests
import shapely

In [2]:
import sys
sys.path.insert(0, '..')

from constantes import (
    RUTA_PROYECTO,
    RUTAS_BLOQUES,
    ruta_entregable,
    exigir_archivo,
    USER_AGENT,
    TIMEZONE,
    CRS_WGS84,
    CRS_METRICO,
    ATRIBUCION_OCM,
    AREAS_ESTUDIO,
    LIMITE_IGAC,
    FECHA_CORTE_LOCAL,
    PREFIJO_CRUDO_OCM,
    cargar_limite_ciudad,
)

# Configuración de OPENCHARGEMAP

In [3]:
# ============================================================================
# Acceso a OpenChargeMap
# ============================================================================
# La clave se lee EXCLUSIVAMENTE de la variable de entorno. No se escribe en
# el cuaderno: todo lo que quede aquí entra al repositorio y queda expuesto.
#
# La clave que estaba escrita en la versión anterior debe revocarse desde
# openchargemap.org: estuvo en texto plano dentro del cuaderno.

OPENCHARGEMAP_API_KEY = os.environ.get("OCM_API_KEY")

OPENCHARGEMAP_URL = "https://api.openchargemap.io/v3/poi/"
OPENCHARGEMAP_MAX_RESULTS = 5000

TIMEOUT_OCM_HTTP = 120
MAX_REINTENTOS_OCM = 3
ESPERA_REINTENTO_OCM = 5

# Parámetros fijos de la consulta. Se declaran aquí y no dentro de la función
# para que queden registrados en los metadatos del congelamiento.
OCM_PARAMETROS_BASE = {
    "output": "json",
    "countrycode": "CO",
    "compact": "false",
    "verbose": "true",
    "includecomments": "false",
}

# Congelamiento del archivo crudo

In [4]:
# Fecha de descarga del archivo crudo
# ----------------------------------------------------------------------------
# NO es el corte. OpenChargeMap no admite consultas históricas, de modo que la
# descarga es necesariamente posterior al estado que el archivo representa. La
# equivalencia entre ambos se verifica sobre el contenido, no se supone.
#
# Origen del dato: fecha de carga del archivo al repositorio de respaldo en
# Drive (02/09/2026), que acota la descarga a ese día o antes, y coincide con
# la fecha de ejecución de la corrida que produjo los entregables de A3.
#
# La fecha de modificación del archivo en disco local (07/09/2026) NO es
# válida: corresponde al día en que se volvió a descargar desde Drive,
# operación que reescribe el archivo y pisa la marca original.
#
# La respuesta de OpenChargeMap no incluye cabecera con la fecha de la
# consulta —a diferencia de Overpass, que trae el bloque osm3s—, de modo que
# el dato no puede leerse del contenido y debe registrarse por fuera.

FECHA_DESCARGA_OCM = "2026-09-02"

# OpenChargeMap no admite consultas históricas: no hay forma de pedirle "el
# estado del registro al 29 de agosto". La única manera de tener un corte es
# guardar la respuesta en disco y no volver a consultarla. El archivo crudo
# ES el corte.
#
# Es la diferencia con A2, donde Overpass sí acepta attic query y el
# congelamiento es una comodidad. Aquí es obligatorio.

def rutas_crudo_ocm(
    fecha_corte_local=FECHA_CORTE_LOCAL,
    fecha_descarga=FECHA_DESCARGA_OCM,
):
    """
    Nombres de los tres archivos del congelamiento.

    El crudo guarda la respuesta; el .sha256 permite comprobar que no se
    modificó; el .meta.json conserva las circunstancias de la descarga, que
    el crudo no contiene.

    El nombre lleva las dos fechas y con etiquetas deliberadamente distintas:

      estado-al-<fecha>  qué contiene el archivo: el instante del mundo que
                         representa. Es el corte.

      bajado-<fecha>     cuándo se trajo de la API. Es una acción, no un
                         estado.

    Van separadas porque no son comparables entre sí ni son dos medidas de lo
    mismo: la primera es un instante con hora y zona horaria detrás, la
    segunda un día de calendario. Lo único que se afirma al ponerlas juntas es
    su orden —la descarga es posterior al estado—, y la prueba de
    compatibilidad es la que sostiene que esa diferencia no altera el
    contenido.
    """

    carpeta = ruta_entregable("A3")

    base = (
        f"{PREFIJO_CRUDO_OCM}_Colombia"
        f"_estado-al-{fecha_corte_local}"
        f"_bajado-{fecha_descarga}"
    )

    return {
        "crudo": carpeta / f"{base}.json",
        "hash": carpeta / f"{base}.sha256",
        "meta": carpeta / f"{base}.meta.json",
    }


def sha256_archivo(ruta):
    """Huella SHA-256 del contenido de un archivo en disco."""

    with open(ruta, "rb") as archivo:
        return hashlib.sha256(archivo.read()).hexdigest()


# ----------------------------------------------------------------------------
# Campos de fecha que declara OpenChargeMap
# ----------------------------------------------------------------------------
# Son la única evidencia disponible sobre la vigencia de cada registro.

CAMPOS_FECHA_OCM = [
    "DateCreated",
    "DateLastStatusUpdate",
    "DateLastVerified",
    "DateLastConfirmed",
]


def verificar_corte_compatible(registros, fecha_corte_local):
    """
    Comprueba que el contenido del crudo sea compatible con el corte.

    En A2 la verificación del corte es directa: se pide un estado histórico y
    se comprueba que ningún elemento fue editado después. Aquí no se puede
    pedir nada: OCM entrega siempre el estado del día de la descarga.

    Lo que sí se puede verificar es la consecuencia observable. Si el archivo
    se descargó DESPUÉS del corte pero ningún registro declara haber sido
    creado ni actualizado después de esa fecha, entonces su contenido es
    indistinguible del que habría devuelto una descarga hecha el día del
    corte. Un registro creado después sobraría; uno actualizado después
    estaría en un estado que al corte no tenía.

    LÍMITE DE LA PRUEBA, que debe declararse en el informe: si OCM modificó
    un registro entre el corte y la descarga sin actualizar ninguno de estos
    campos, esta verificación no lo detecta. No hay forma de detectarlo con
    los datos que la fuente publica. La evidencia disponible es la que hay.

    Devuelve las fechas máximas halladas, que se registran como constancia.
    """

    # Fin del día del corte en hora de Bogotá, expresado en UTC, que es como
    # OpenChargeMap publica sus fechas.
    limite = datetime.fromisoformat(
        f"{fecha_corte_local}T23:59:59"
    ).replace(tzinfo=TIMEZONE).astimezone(ZoneInfo("UTC"))

    maximos = {}

    posteriores = []

    for campo in CAMPOS_FECHA_OCM:

        fechas = []

        for registro in registros:

            valor = registro.get(campo)

            if not valor:
                continue

            fecha = datetime.fromisoformat(
                valor.replace("Z", "+00:00")
            )

            fechas.append(fecha)

            if fecha > limite:
                posteriores.append(
                    (registro.get("ID"), campo, valor)
                )

        maximos[campo] = (
            max(fechas).isoformat() if fechas else None
        )

    if posteriores:

        detalle = ", ".join(
            f"ID {i} ({c} = {v})"
            for i, c, v in posteriores[:5]
        )

        raise RuntimeError(
            f"El archivo crudo contiene {len(posteriores)} fechas "
            f"posteriores al corte {fecha_corte_local}. Su contenido NO es "
            "equivalente al del día del corte. Primeros casos: "
            f"{detalle}"
        )

    print(
        f"  Corte compatible: ninguna de las {len(CAMPOS_FECHA_OCM)} fechas "
        f"declaradas supera el {fecha_corte_local}."
    )

    for campo, maximo in maximos.items():
        print(f"    {campo}: máximo {maximo}")

    return maximos

# Descarga nacional desde OPENCHARGEMAP

In [5]:
def leer_crudo_congelado_ocm(rutas, fecha_corte_local):
    """
    Reconstruye la descarga a partir del archivo crudo congelado, en lugar de
    volver a consultar la API.

    Antes de usarlo comprueba que el archivo no se haya modificado desde que
    se congeló, y que su contenido siga siendo compatible con el corte
    vigente. Un crudo que no supere ambas cosas no es reutilizable.

    El .meta.json puede no existir la primera vez: si el archivo se congeló a
    mano —restituido desde una copia de respaldo, por ejemplo— los dos
    acompañantes se generan aquí, sin tocar la red.
    """

    print("\n" + "=" * 80)
    print("A3 — LECTURA DEL ARCHIVO CRUDO CONGELADO")
    print("=" * 80)

    print(f"\nArchivo: {rutas['crudo']}")

    with open(rutas["crudo"], encoding="utf-8") as archivo:
        datos = json.load(archivo)

    if not isinstance(datos, list) or len(datos) == 0:

        raise RuntimeError(
            f"El archivo crudo {rutas['crudo']} no contiene una lista de "
            "registros utilizable."
        )

    hash_real = sha256_archivo(rutas["crudo"])

    # ------------------------------------------------------------------------
    # Huella
    # ------------------------------------------------------------------------

    if os.path.exists(rutas["hash"]):

        with open(rutas["hash"], encoding="utf-8") as archivo:
            hash_declarado = archivo.read().split()[0].strip()

        if hash_real != hash_declarado:

            raise RuntimeError(
                "El archivo crudo fue modificado después de congelarse. "
                f"SHA-256 declarado {hash_declarado}, recalculado {hash_real}. "
                "Restituya el archivo original desde la copia de respaldo."
            )

        print(f"  SHA-256 verificado: {hash_real[:16]}")

    else:

        with open(rutas["hash"], "w", encoding="utf-8") as archivo:
            archivo.write(
                f"{hash_real}  {os.path.basename(rutas['crudo'])}\n"
            )

        print(
            f"  SHA-256 calculado y registrado por primera vez: "
            f"{hash_real[:16]}"
        )

    # ------------------------------------------------------------------------
    # Compatibilidad con el corte
    # ------------------------------------------------------------------------

    maximos = verificar_corte_compatible(datos, fecha_corte_local)

    # ------------------------------------------------------------------------
    # Metadatos
    # ------------------------------------------------------------------------

    if os.path.exists(rutas["meta"]):

        with open(rutas["meta"], encoding="utf-8") as archivo:
            meta = json.load(archivo)

        if meta.get("fecha_corte_local") != fecha_corte_local:

            raise RuntimeError(
                f"El crudo corresponde al corte "
                f"{meta.get('fecha_corte_local')!r} y la configuración "
                f"vigente pide {fecha_corte_local!r}."
            )

    else:

        # Congelado a mano: se levanta la ficha ahora. Lo que no se puede
        # reconstruir se declara como desconocido en lugar de inventarse.

        meta = {
            "fuente": "OpenChargeMap POI API v3",
            "pais": "Colombia",
            "url": OPENCHARGEMAP_URL,
            "parametros": OCM_PARAMETROS_BASE,
            "fecha_corte_local": fecha_corte_local,
            "fecha_descarga_declarada": FECHA_DESCARGA_OCM,
            "fecha_inicio": None,
            "fecha_fin": None,
            "duracion_segundos": None,
            "n_registros": len(datos),
            "truncado": len(datos) >= OPENCHARGEMAP_MAX_RESULTS,
            "limite_solicitado": OPENCHARGEMAP_MAX_RESULTS,
            "fechas_maximas_declaradas": maximos,
            "origen_bytes": (
                "archivo reserializado con indentación; NO son los bytes "
                "originales de la API. El hash garantiza integridad desde "
                "este momento, no identidad con la respuesta del servidor."
            ),
            "congelado_a_mano": True,
            "nota_procedencia": (
                "Archivo restituido desde la copia de respaldo del proyecto. "
                "La hora exacta de la descarga no quedó registrada; la fecha "
                "se acota por la carga del archivo al repositorio de respaldo."
            ),
            "atribucion": ATRIBUCION_OCM,
        }

        with open(rutas["meta"], "w", encoding="utf-8") as archivo:
            json.dump(meta, archivo, ensure_ascii=False, indent=2)

        print(f"  Metadatos generados: {rutas['meta']}")

    print(f"  {len(datos)} registros leídos sin consultar la API.")

    return {
        "datos": datos,
        "origen": "archivo crudo congelado",
        "fecha_inicio": meta.get("fecha_inicio"),
        "fecha_fin": meta.get("fecha_fin"),
        "duracion_segundos": meta.get("duracion_segundos"),
        "parametros": meta.get("parametros"),
        "n_registros": len(datos),
        "truncado": meta.get("truncado", False),
        "intentos": [],
        "ruta_crudo": str(rutas["crudo"]),
        "hash_crudo": hash_real,
        "fecha_corte_local": fecha_corte_local,
        "fecha_descarga_declarada": meta.get("fecha_descarga_declarada"),
        "fechas_maximas_declaradas": maximos,
    }

def descargar_ocm_colombia(
    fecha_corte_local=FECHA_CORTE_LOCAL,
    fecha_descarga=FECHA_DESCARGA_OCM,
):
    """
    Obtiene los registros de OpenChargeMap para Colombia.

    Tiene dos vías y siempre prefiere la primera:

    1. Si el archivo crudo congelado existe en disco, se lee de ahí previa
       verificación de su huella. Ninguna ejecución posterior consulta la API.

    2. Si no existe, se descarga y se congela: se escriben los bytes exactos
       de la respuesta, su SHA-256 y un archivo de metadatos.

    La diferencia con A2 es que aquí la primera vía no es una comodidad sino
    la única forma de tener un corte. Overpass admite attic query y puede
    entregar el estado de cualquier fecha pasada; OpenChargeMap entrega
    siempre el estado de hoy. El archivo crudo ES el corte, y si se pierde no
    se puede volver a obtener.
    """

    rutas = rutas_crudo_ocm(fecha_corte_local, fecha_descarga)

    # ========================================================================
    # Vía 1 — el crudo ya está congelado
    # ========================================================================

    if os.path.exists(rutas["crudo"]):

        return leer_crudo_congelado_ocm(rutas, fecha_corte_local)

    # ========================================================================
    # Vía 2 — no hay crudo: se descarga y se congela
    # ========================================================================

    print("\n" + "=" * 80)
    print("A3 — DESCARGA NACIONAL DESDE OPENCHARGEMAP")
    print("=" * 80)

    print(
        f"\n⚠ No se encontró el archivo crudo congelado:\n"
        f"  {rutas['crudo']}\n"
        f"  Se procederá a descargar de la API. ATENCIÓN: la descarga traerá\n"
        f"  el estado de HOY, no el del corte {fecha_corte_local}, y la\n"
        f"  verificación de compatibilidad fallará si el registro cambió.\n"
        f"  Si el crudo del corte existe en otra carpeta, cancele y muévalo\n"
        f"  antes de continuar."
    )

    # La clave solo se exige aquí: con el crudo congelado el cuaderno debe
    # poder ejecutarse de principio a fin sin clave y sin red.

    if not OPENCHARGEMAP_API_KEY:

        raise RuntimeError(
            "No hay clave de OpenChargeMap y no existe el archivo crudo "
            "congelado. Fije la variable de entorno OCM_API_KEY, o restituya "
            f"el archivo {rutas['crudo']}."
        )

    fecha_inicio = datetime.now(TIMEZONE)

    parametros = dict(OCM_PARAMETROS_BASE)

    parametros["maxresults"] = OPENCHARGEMAP_MAX_RESULTS

    # La clave se inyecta solo en la petición, nunca en OCM_PARAMETROS_BASE:
    # ese diccionario se guarda en el .meta.json y quedaría expuesta.
    parametros_con_clave = dict(parametros)

    parametros_con_clave["key"] = OPENCHARGEMAP_API_KEY

    headers = {"User-Agent": USER_AGENT}

    ultimo_error = None

    intentos = []

    for intento in range(1, MAX_REINTENTOS_OCM + 1):

        print(f"\nIntento {intento}/{MAX_REINTENTOS_OCM}")

        try:

            respuesta = requests.get(
                OPENCHARGEMAP_URL,
                params=parametros_con_clave,
                headers=headers,
                timeout=TIMEOUT_OCM_HTTP,
            )

            print(f"Código HTTP: {respuesta.status_code}")

            respuesta.raise_for_status()

            datos = respuesta.json()

            if not isinstance(datos, list):

                raise RuntimeError(
                    "OpenChargeMap no devolvió una lista de registros: "
                    f"{type(datos)}"
                )

            if len(datos) == 0:

                raise RuntimeError(
                    "OpenChargeMap devolvió cero registros."
                )

            # ----------------------------------------------------------------
            # Truncamiento
            # ----------------------------------------------------------------
            # Si vuelve exactamente el máximo solicitado, el inventario puede
            # estar cortado sin que la API lo advierta.

            truncado = len(datos) >= OPENCHARGEMAP_MAX_RESULTS

            if truncado:

                raise RuntimeError(
                    f"Se recuperaron exactamente {len(datos)} registros, "
                    "igual al límite solicitado. El inventario puede estar "
                    "truncado. Aumente OPENCHARGEMAP_MAX_RESULTS y repita. "
                    "No se congela un archivo posiblemente incompleto."
                )

            fecha_fin = datetime.now(TIMEZONE)

            intentos.append({"intento": intento, "resultado": "exitoso"})

            maximos = verificar_corte_compatible(datos, fecha_corte_local)

            # ----------------------------------------------------------------
            # Congelamiento
            # ----------------------------------------------------------------
            # Se escriben los bytes tal como llegaron, sin reserializar: el
            # hash debe corresponder a lo que el servidor envió y no a lo que
            # Python decidió imprimir.

            os.makedirs(os.path.dirname(rutas["crudo"]), exist_ok=True)

            with open(rutas["crudo"], "wb") as archivo:
                archivo.write(respuesta.content)

            hash_crudo = sha256_archivo(rutas["crudo"])

            with open(rutas["hash"], "w", encoding="utf-8") as archivo:
                archivo.write(
                    f"{hash_crudo}  {os.path.basename(rutas['crudo'])}\n"
                )

            meta = {
                "fuente": "OpenChargeMap POI API v3",
                "pais": "Colombia",
                "url": OPENCHARGEMAP_URL,
                "parametros": parametros,
                "fecha_corte_local": fecha_corte_local,
                "fecha_descarga_declarada": fecha_descarga,
                "fecha_inicio": fecha_inicio.isoformat(),
                "fecha_fin": fecha_fin.isoformat(),
                "duracion_segundos": (
                    fecha_fin - fecha_inicio
                ).total_seconds(),
                "n_registros": len(datos),
                "truncado": truncado,
                "limite_solicitado": OPENCHARGEMAP_MAX_RESULTS,
                "fechas_maximas_declaradas": maximos,
                "origen_bytes": "respuesta original de la API, sin reserializar",
                "atribucion": ATRIBUCION_OCM,
            }

            with open(rutas["meta"], "w", encoding="utf-8") as archivo:
                json.dump(meta, archivo, ensure_ascii=False, indent=2)

            print(f"✓ Registros recuperados: {len(datos)}")
            print(f"  Crudo congelado en: {rutas['crudo']}")
            print(f"  SHA-256: {hash_crudo[:16]}")

            return {
                "datos": datos,
                "origen": "descarga nueva, congelada en disco",
                "fecha_inicio": fecha_inicio.isoformat(),
                "fecha_fin": fecha_fin.isoformat(),
                "duracion_segundos": (
                    fecha_fin - fecha_inicio
                ).total_seconds(),
                "parametros": parametros,
                "n_registros": len(datos),
                "truncado": truncado,
                "intentos": intentos,
                "ruta_crudo": str(rutas["crudo"]),
                "hash_crudo": hash_crudo,
                "fecha_corte_local": fecha_corte_local,
                "fecha_descarga_declarada": fecha_descarga,
                "fechas_maximas_declaradas": maximos,
            }

        except Exception as error:

            print(f"✗ Error: {error}")

            intentos.append({
                "intento": intento,
                "resultado": f"fallo: {error}",
            })

            ultimo_error = error

            if intento < MAX_REINTENTOS_OCM:
                time.sleep(ESPERA_REINTENTO_OCM)

    raise RuntimeError(
        "No fue posible descargar desde OpenChargeMap. "
        f"Último error: {ultimo_error}"
    )

# Extracción y normalizacion de atributos

In [6]:
def _titulo(objeto):
    """
    Extrae el campo Title de un objeto anidado de OpenChargeMap.
    Devuelve None si el objeto no existe o no es un diccionario.
    """

    if isinstance(objeto, dict):
        return objeto.get("Title")

    return None


def _numero(valor):
    """
    Convierte a float, devolviendo None si no es convertible.
    """

    if valor is None:
        return None

    try:
        return float(valor)

    except (TypeError, ValueError):
        return None


def transformar_registros_ocm(
    datos,
    fecha_descarga,
):
    """
    Transforma la respuesta de OpenChargeMap en DOS tablas planas.

    Correcciones respecto de la versión anterior, según la revisión
    del Bloque A:

    1. 'operador' quedaba como el diccionario crudo de la API, con
       sitio web, teléfonos e identificadores dentro de una celda.
       Ahora son columnas planas.

    2. 'pais' tenía el mismo problema: AddressInfo.Country es un
       diccionario, no una cadena. Se separa en nombre y código ISO.

    3. 'tipo_acceso' salía vacío en los 26 registros porque se leía
       AddressInfo.AccessComments, un campo de texto libre que casi
       nadie llena. El tipo de acceso real está en UsageType, en las
       banderas IsPayAtLocation, IsMembershipRequired e
       IsAccessKeyRequired. AccessComments se conserva aparte, con
       su nombre correcto, para no perder el dato.

    4. Los conectores ya no se colapsan en una cadena separada por
       barras. Se produce una segunda tabla con una fila por
       conexión, que conserva la correspondencia entre conector,
       potencia, corriente y nivel. Para el ruteo esa
       correspondencia es lo esencial: no basta saber que hay un
       CCS y que hay 50 kW, hay que saber que el CCS es el de 50 kW.

    5. Se incorporan campos de vigencia adicionales
       (DateLastStatusUpdate, DateCreated, IsRecentlyVerified,
       DataQualityLevel) y la bandera IsOperational, para poder
       verificar si 'Operational' refleja la realidad o es el valor
       por defecto de la fuente.

    6. Se devuelve una auditoría del recorrido. Los conteos se
       producen dentro del mismo bucle que genera las filas, de modo
       que el balance sea una verificación real y no una resta hecha
       después. Es la disciplina que dejó la revisión del Bloque A:
       todo número importante se comprueba por un camino distinto al
       que lo produjo.

    Returns
    -------
    tuple(pandas.DataFrame, pandas.DataFrame, dict)
        Tabla de estaciones, tabla de conexiones y auditoría del
        recorrido.
    """

    filas_estaciones = []

    filas_conexiones = []

    # ------------------------------------------------------------------------
    # Contadores del balance
    # ------------------------------------------------------------------------

    registros_sin_id = 0

    registros_sin_coordenadas = 0

    conexiones_no_validas = 0

    for punto in datos:

        # ---------------------------------------------------------------
        # Descarte: elemento inutilizable
        # ---------------------------------------------------------------
        # Sin identificador no hay forma de referirse al registro ni de
        # ligarlo con su tabla de conexiones.

        if not isinstance(punto, dict):
            registros_sin_id += 1
            continue

        ocm_id = punto.get("ID")

        if ocm_id is None:
            registros_sin_id += 1
            continue

        # ---------------------------------------------------------------
        # Dirección
        # ---------------------------------------------------------------

        address = punto.get("AddressInfo", {})

        if not isinstance(address, dict):
            address = {}

        pais_obj = address.get("Country")

        if not isinstance(pais_obj, dict):
            pais_obj = {}

        # ---------------------------------------------------------------
        # Coordenadas
        # ---------------------------------------------------------------
        # Las estaciones sin coordenadas SÍ se conservan en la tabla: el
        # registro existe y sus atributos son válidos. Se cuentan aparte
        # porque no podrán ubicarse en el mapa y, por tanto, ninguna
        # entrará a una ciudad al aplicar el recorte espacial.

        latitud = _numero(address.get("Latitude"))

        longitud = _numero(address.get("Longitude"))

        if latitud is None or longitud is None:
            registros_sin_coordenadas += 1

        # ---------------------------------------------------------------
        # Operador: columnas planas
        # ---------------------------------------------------------------

        operador_obj = punto.get("OperatorInfo")

        if not isinstance(operador_obj, dict):
            operador_obj = {}

        # ---------------------------------------------------------------
        # Tipo de uso y tipo de acceso
        # ---------------------------------------------------------------

        uso_obj = punto.get("UsageType")

        if not isinstance(uso_obj, dict):
            uso_obj = {}

        # ---------------------------------------------------------------
        # Estado operativo
        # ---------------------------------------------------------------

        estado_obj = punto.get("StatusType")

        if not isinstance(estado_obj, dict):
            estado_obj = {}

        # ---------------------------------------------------------------
        # Conexiones: una fila por conector
        # ---------------------------------------------------------------

        conexiones = punto.get("Connections", [])

        if not isinstance(conexiones, list):
            conexiones = []

        potencias = []

        conectores = []

        tiene_dc = False

        for indice, conexion in enumerate(conexiones):

            if not isinstance(conexion, dict):
                conexiones_no_validas += 1
                continue

            tipo_obj = conexion.get("ConnectionType")

            if not isinstance(tipo_obj, dict):
                tipo_obj = {}

            nivel_obj = conexion.get("Level")

            if not isinstance(nivel_obj, dict):
                nivel_obj = {}

            corriente_obj = conexion.get("CurrentType")

            if not isinstance(corriente_obj, dict):
                corriente_obj = {}

            estado_conexion_obj = conexion.get("StatusType")

            if not isinstance(estado_conexion_obj, dict):
                estado_conexion_obj = {}

            potencia = _numero(
                conexion.get("PowerKW")
            )

            titulo_conector = tipo_obj.get("Title")

            corriente = corriente_obj.get("Title")

            if potencia is not None:
                potencias.append(potencia)

            if titulo_conector:
                conectores.append(str(titulo_conector))

            if corriente and "DC" in str(corriente).upper():
                tiene_dc = True

            filas_conexiones.append({

                "ocm_id": ocm_id,

                "conexion_indice": indice,

                "conexion_id": conexion.get("ID"),

                "conector": titulo_conector,

                "conector_nombre_formal": (
                    tipo_obj.get("FormalName")
                ),

                "conector_obsoleto": (
                    tipo_obj.get("IsObsolete")
                ),

                "conector_descontinuado": (
                    tipo_obj.get("IsDiscontinued")
                ),

                "potencia_kw": potencia,

                "amperaje": _numero(
                    conexion.get("Amps")
                ),

                "voltaje": _numero(
                    conexion.get("Voltage")
                ),

                "tipo_corriente": corriente,

                "nivel": nivel_obj.get("Title"),

                "nivel_carga_rapida": (
                    nivel_obj.get(
                        "IsFastChargeCapable"
                    )
                ),

                "cantidad": conexion.get("Quantity"),

                "estado_conexion": (
                    estado_conexion_obj.get("Title")
                ),

                "estado_conexion_operativa": (
                    estado_conexion_obj.get(
                        "IsOperational"
                    )
                ),

                "fecha_descarga": fecha_descarga,

                "fuente": "OpenChargeMap",
            })

        # ---------------------------------------------------------------
        # Registro de estación
        # ---------------------------------------------------------------

        filas_estaciones.append({

            # --- identificación ---

            "ocm_id": ocm_id,

            "uuid": punto.get("UUID"),

            # --- ubicación ---

            "nombre": address.get("Title"),

            "direccion": address.get("AddressLine1"),

            "ciudad": address.get("Town"),

            "estado_provincia": (
                address.get("StateOrProvince")
            ),

            "codigo_postal": address.get("Postcode"),

            "pais": pais_obj.get("Title"),

            "pais_iso": pais_obj.get("ISOCode"),

            "latitud": latitud,

            "longitud": longitud,

            # --- operador (columnas planas) ---

            "operador": operador_obj.get("Title"),

            "operador_id": operador_obj.get("ID"),

            "operador_web": (
                operador_obj.get("WebsiteURL")
            ),

            "operador_es_particular": (
                operador_obj.get(
                    "IsPrivateIndividual"
                )
            ),

            # --- tipo de uso ---

            "tipo_uso": uso_obj.get("Title"),

            "tipo_uso_id": uso_obj.get("ID"),

            # --- tipo de acceso (corregido) ---

            "acceso_pago_en_sitio": (
                uso_obj.get("IsPayAtLocation")
            ),

            "acceso_requiere_membresia": (
                uso_obj.get("IsMembershipRequired")
            ),

            "acceso_requiere_llave": (
                uso_obj.get("IsAccessKeyRequired")
            ),

            "acceso_comentarios": (
                address.get("AccessComments")
            ),

            "costo_uso": punto.get("UsageCost"),

            # --- estado operativo ---

            "estado_operativo": (
                estado_obj.get("Title")
            ),

            "estado_operativo_id": (
                estado_obj.get("ID")
            ),

            "estado_es_operativo": (
                estado_obj.get("IsOperational")
            ),

            # --- resumen de conexiones ---

            "n_conexiones": len(conexiones),

            "n_puntos_declarados": (
                punto.get("NumberOfPoints")
            ),

            "conectores_lista": (
                " | ".join(
                    dict.fromkeys(conectores)
                )
                if conectores
                else None
            ),

            "potencia_kw_max": (
                max(potencias)
                if potencias
                else None
            ),

            "potencia_kw_min": (
                min(potencias)
                if potencias
                else None
            ),

            "tiene_carga_dc": tiene_dc,

            # --- vigencia ---

            "fecha_ultima_verificacion": (
                punto.get("DateLastVerified")
            ),

            "fecha_ultimo_cambio_estado": (
                punto.get("DateLastStatusUpdate")
            ),

            "fecha_creacion": (
                punto.get("DateCreated")
            ),

            "verificado_recientemente": (
                punto.get("IsRecentlyVerified")
            ),

            "nivel_calidad_dato": (
                punto.get("DataQualityLevel")
            ),

            # --- procedencia ---

            "proveedor_datos": _titulo(
                punto.get("DataProvider")
            ),

            "fecha_descarga": fecha_descarga,

            "fuente": "OpenChargeMap",

            "licencia": (
                "OpenChargeMap — pendiente C3"
            ),
        })

    df_estaciones = pd.DataFrame(filas_estaciones)

    df_conexiones = pd.DataFrame(filas_conexiones)

    if df_estaciones.empty:

        raise RuntimeError(
            "La transformación produjo una tabla vacía."
        )

    # ------------------------------------------------------------------------
    # Auditoría del recorrido
    # ------------------------------------------------------------------------
    # El balance que se verifica es:
    #
    #     elementos_recibidos = registros_transformados + registros_sin_id
    #
    # 'registros_sin_coordenadas' NO entra en esa suma: esas estaciones están
    # dentro de registros_transformados. Son un diagnóstico aparte, no un
    # destino del recorrido.

    auditoria = {

        "elementos_recibidos": len(datos),

        "registros_transformados": len(df_estaciones),

        "registros_sin_id": registros_sin_id,

        "conexiones_generadas": len(df_conexiones),

        "conexiones_no_validas": conexiones_no_validas,

        "registros_sin_coordenadas": registros_sin_coordenadas,

        "estaciones_sin_conexiones": int(
            (df_estaciones["n_conexiones"] == 0).sum()
        ),
    }

    print(
        f"\n✓ Estaciones: {df_estaciones.shape[0]} filas "
        f"× {df_estaciones.shape[1]} columnas"
    )

    print(
        f"✓ Conexiones: {df_conexiones.shape[0]} filas "
        f"× {df_conexiones.shape[1]} columnas"
    )

    print(
        f"\n  Balance: {auditoria['elementos_recibidos']} recibidos = "
        f"{auditoria['registros_transformados']} transformados + "
        f"{auditoria['registros_sin_id']} sin identificador"
    )

    print(
        f"  Estaciones sin coordenadas utilizables: "
        f"{auditoria['registros_sin_coordenadas']}"
    )

    print(
        f"  Estaciones sin ninguna conexión declarada: "
        f"{auditoria['estaciones_sin_conexiones']}"
    )

    if conexiones_no_validas:
        print(
            f"  Conexiones descartadas por formato: "
            f"{conexiones_no_validas}"
        )

    return df_estaciones, df_conexiones, auditoria

# Validación estructural

In [7]:
# ============================================================================
# Caja envolvente del territorio colombiano
# ============================================================================
# QUÉ ES
# Los cuatro números que encierran a Colombia en un rectángulo: latitud de
# -4,5° (sur del Amazonas) a 13,5° (norte de La Guajira, con las islas de San
# Andrés y Providencia dentro), y longitud de -82° (occidente de San Andrés)
# a -66° (oriente de la Orinoquía). Todo punto del país cae dentro.
#
# PARA QUÉ SE USA
# Para detectar coordenadas invertidas, que es el error más común en datos
# geográficos: escribir la longitud en el campo de la latitud y viceversa.
# Una estación de Bogotá tiene latitud 4,6 y longitud -74,1; si se invierten,
# queda en latitud -74,1 y longitud 4,6, que es un punto del océano al sur de
# África. Fuera de la caja, y por tanto detectable.
#
# POR QUÉ AQUÍ Y NO DESPUÉS DEL RECORTE
# Porque el recorte por límite municipal también descartaría ese registro,
# pero sin decir por qué. La estación desaparecería del inventario en
# silencio y el conteo bajaría de 119 a 118 sin explicación. Comprobarlo
# antes convierte un fallo invisible en un error que nombra el registro
# culpable y muestra sus coordenadas.
#
# LO QUE NO ES
# No sustituye al recorte por límite municipal. Un punto puede estar dentro
# de esta caja y no estar en Bogotá ni en Medellín —de hecho la mayoría de
# los 119 registros nacionales están en esa situación—. La caja solo separa
# lo geográficamente posible de lo imposible.
#
# HOLGURA
# Los límites llevan margen sobre las fronteras reales a propósito. No se
# trata de decidir si un punto está en Colombia, sino de detectar disparates.

RANGO_COLOMBIA = {
    "lat_min": -4.5,
    "lat_max": 13.5,
    "lon_min": -82.0,
    "lon_max": -66.0,
}

def validar_inventario_ocm(
    df_estaciones,
    df_conexiones,
    auditoria=None,
):
    """
    Validación estructural del inventario nacional.

    Comprueba lo que debe cumplirse antes de cualquier recorte espacial.
    Las fallas que aquí se detectan son de estructura, no de contenido:
    un identificador duplicado o una coordenada invertida invalidan todo
    lo que venga después, y conviene que revienten temprano, cuando
    todavía se puede saber qué registro las causó.

    Devuelve una tabla de comprobaciones y levanta excepción ante
    cualquier falla que impida continuar.
    """

    comprobaciones = []

    def registrar(nombre, valor, detalle=""):

        comprobaciones.append({
            "comprobacion": nombre,
            "valor": valor,
            "detalle": detalle,
        })

    # ------------------------------------------------------------------------
    # 1. Tablas no vacías
    # ------------------------------------------------------------------------

    if df_estaciones.empty:
        raise AssertionError(
            "La tabla de estaciones está vacía."
        )

    registrar(
        "Estaciones",
        len(df_estaciones),
        f"{df_estaciones.shape[1]} columnas",
    )

    registrar(
        "Conexiones",
        len(df_conexiones),
        (
            f"{df_conexiones.shape[1]} columnas"
            if not df_conexiones.empty
            else "sin conexiones declaradas"
        ),
    )

    # ------------------------------------------------------------------------
    # 2. Identificador único
    # ------------------------------------------------------------------------

    duplicados = int(
        df_estaciones["ocm_id"].duplicated().sum()
    )

    if duplicados:

        ejemplos = (
            df_estaciones.loc[
                df_estaciones["ocm_id"].duplicated(keep=False),
                "ocm_id",
            ]
            .unique()[:5]
            .tolist()
        )

        raise AssertionError(
            f"Hay {duplicados} ocm_id duplicados. Ejemplos: {ejemplos}"
        )

    registrar(
        "ocm_id únicos",
        int(df_estaciones["ocm_id"].nunique()),
        "sin duplicados",
    )

    # ------------------------------------------------------------------------
    # 3. Integridad referencial estaciones ↔ conexiones
    # ------------------------------------------------------------------------
    # La versión anterior afirmaba "sin huérfanas" sin comprobarlo. Con las
    # conexiones en tabla aparte, una conexión cuyo ocm_id no exista entre
    # las estaciones sería un registro imposible de interpretar.

    if not df_conexiones.empty:

        ids_estaciones = set(df_estaciones["ocm_id"])

        ids_conexiones = set(df_conexiones["ocm_id"])

        huerfanas = ids_conexiones - ids_estaciones

        if huerfanas:

            raise AssertionError(
                f"Hay {len(huerfanas)} conexiones cuyo ocm_id no existe en "
                f"la tabla de estaciones. Ejemplos: {sorted(huerfanas)[:5]}"
            )

        registrar(
            "Integridad referencial",
            len(ids_conexiones),
            "todo ocm_id de conexiones existe en estaciones",
        )

        # Verificación independiente de la potencia máxima: se recalcula
        # desde las conexiones y se contrasta con el valor guardado en la
        # tabla de estaciones, que se produjo por otro camino.

        recalculada = (
            df_conexiones
            .groupby("ocm_id")["potencia_kw"]
            .max()
        )

        declarada = (
            df_estaciones
            .set_index("ocm_id")["potencia_kw_max"]
            .dropna()
        )

        comunes = declarada.index.intersection(recalculada.index)

        discrepancias = int(
            (
                declarada.loc[comunes] != recalculada.loc[comunes]
            ).sum()
        )

        if discrepancias:

            raise AssertionError(
                f"En {discrepancias} estaciones la potencia máxima no "
                "coincide con el recálculo desde la tabla de conexiones."
            )

        registrar(
            "Verificación independiente: potencia máxima",
            len(comunes),
            "recalculada desde conexiones, coincide",
        )

    # ------------------------------------------------------------------------
    # 4. Coordenadas
    # ------------------------------------------------------------------------

    sin_coordenadas = int(
        df_estaciones[["latitud", "longitud"]]
        .isna()
        .any(axis=1)
        .sum()
    )

    registrar(
        "Estaciones sin coordenadas",
        sin_coordenadas,
        "se conservan en la tabla; no entrarán a ninguna ciudad",
    )

    con_coordenadas = df_estaciones.dropna(
        subset=["latitud", "longitud"]
    )

    fuera_de_rango = con_coordenadas[
        (con_coordenadas["latitud"] < RANGO_COLOMBIA["lat_min"])
        | (con_coordenadas["latitud"] > RANGO_COLOMBIA["lat_max"])
        | (con_coordenadas["longitud"] < RANGO_COLOMBIA["lon_min"])
        | (con_coordenadas["longitud"] > RANGO_COLOMBIA["lon_max"])
    ]

    if len(fuera_de_rango):

        ejemplos = (
            fuera_de_rango[["ocm_id", "nombre", "latitud", "longitud"]]
            .head(5)
            .to_dict("records")
        )

        raise AssertionError(
            f"{len(fuera_de_rango)} estaciones con coordenadas fuera del "
            "territorio colombiano. La consulta se hizo con countrycode=CO, "
            "de modo que esto indica coordenadas invertidas o corruptas, no "
            "estaciones extranjeras. Ejemplos: "
            f"{ejemplos}"
        )

    registrar(
        "Coordenadas dentro del territorio",
        len(con_coordenadas),
        (
            f"lat [{RANGO_COLOMBIA['lat_min']}, "
            f"{RANGO_COLOMBIA['lat_max']}], "
            f"lon [{RANGO_COLOMBIA['lon_min']}, "
            f"{RANGO_COLOMBIA['lon_max']}]"
        ),
    )

    # ------------------------------------------------------------------------
    # 5. Balance de la transformación
    # ------------------------------------------------------------------------
    # Los conteos se produjeron dentro del bucle que generó las filas. Que
    # sumen es una verificación real; comprobarlo aquí con una resta a
    # posteriori no lo sería.

    if auditoria is not None:

        suma = (
            auditoria["registros_transformados"]
            + auditoria["registros_sin_id"]
        )

        if suma != auditoria["elementos_recibidos"]:

            raise AssertionError(
                "El balance de la transformación no cuadra. Recibidos "
                f"{auditoria['elementos_recibidos']}, suma de destinos "
                f"{suma}. Hay elementos que el código no está contabilizando."
            )

        registrar(
            "Balance de la transformación",
            auditoria["elementos_recibidos"],
            (
                f"{auditoria['elementos_recibidos']} = "
                f"{auditoria['registros_transformados']} transformados + "
                f"{auditoria['registros_sin_id']} sin identificador"
            ),
        )

        # Contraste entre el conteo del bucle y el de la tabla ya construida:
        # dos caminos distintos hacia el mismo número.

        sin_coord_tabla = sin_coordenadas

        if auditoria["registros_sin_coordenadas"] != sin_coord_tabla:

            raise AssertionError(
                "El conteo de estaciones sin coordenadas difiere entre el "
                f"recorrido ({auditoria['registros_sin_coordenadas']}) y la "
                f"tabla final ({sin_coord_tabla})."
            )

        registrar(
            "Sin coordenadas: recorrido vs tabla",
            sin_coord_tabla,
            "ambos caminos coinciden",
        )

    return pd.DataFrame(comprobaciones)

# Construcción de la información geografica

In [8]:
def convertir_a_geodataframe(df_estaciones):
    """
    Construye la capa de puntos a partir de las columnas latitud y longitud.

    Separa de manera explícita las estaciones que no tienen coordenadas, en
    lugar de dejarlas pasar con geometría vacía. La diferencia importa: una
    geometría vacía no falla al crearse, pero tampoco intersecta ningún
    polígono, de modo que el recorte espacial la descartaría en silencio y el
    inventario perdería registros sin que nada lo advirtiera.

    Devuelve la capa de puntos y la tabla de las estaciones sin ubicación,
    que no son un residuo del procesamiento: son registros válidos de
    OpenChargeMap a los que la fuente no asignó coordenadas, y su conteo es
    un dato sobre la calidad del inventario nacional.

    Returns
    -------
    tuple(geopandas.GeoDataFrame, pandas.DataFrame)
        Capa de puntos en EPSG:4326 y tabla de estaciones sin coordenadas.
    """

    sin_coordenadas = df_estaciones[
        df_estaciones[["latitud", "longitud"]].isna().any(axis=1)
    ].copy()

    con_coordenadas = df_estaciones.dropna(
        subset=["latitud", "longitud"]
    ).copy()

    if con_coordenadas.empty:

        raise RuntimeError(
            "Ninguna estación tiene coordenadas utilizables: no es posible "
            "construir la capa de puntos."
        )

    gdf = gpd.GeoDataFrame(
        con_coordenadas,
        geometry=gpd.points_from_xy(
            con_coordenadas["longitud"],
            con_coordenadas["latitud"],
        ),
        crs=CRS_WGS84,
    ).reset_index(drop=True)

    # ------------------------------------------------------------------------
    # Verificación independiente
    # ------------------------------------------------------------------------
    # Se comprueba que la geometría construida corresponda a las columnas que
    # la produjeron, leyendo de vuelta las coordenadas desde el punto. El
    # orden de los argumentos de points_from_xy es (x, y), es decir
    # (longitud, latitud): invertirlo es un error que no produce excepción y
    # que desplazaría todas las estaciones a otro lugar del planeta.

    desfase_x = (gdf.geometry.x - gdf["longitud"]).abs().max()

    desfase_y = (gdf.geometry.y - gdf["latitud"]).abs().max()

    if desfase_x > 1e-9 or desfase_y > 1e-9:

        raise AssertionError(
            "La geometría no corresponde a las columnas de coordenadas. "
            f"Desfase máximo en x {desfase_x}, en y {desfase_y}. Revise el "
            "orden de los argumentos de points_from_xy."
        )

    if not gdf.geometry.is_valid.all():

        raise AssertionError(
            "Hay geometrías inválidas en la capa de puntos."
        )

    print(
        f"\n✓ Capa de puntos: {len(gdf)} estaciones en {CRS_WGS84}"
    )

    print(
        "  Geometría verificada contra las columnas latitud/longitud."
    )

    if not sin_coordenadas.empty:

        print(
            f"  ⚠ {len(sin_coordenadas)} estaciones quedan fuera de la capa "
            "por no tener coordenadas en la fuente."
        )

    return gdf, sin_coordenadas

# Carga de los límites oficiales del IGAC

In [9]:
def cargar_limites_estudio():
    """
    Carga los límites municipales de las dos ciudades de estudio.

    QUÉ CAMBIÓ Y POR QUÉ
    La versión anterior leía los archivos por su cuenta, con su propio código,
    sin disolver las geometrías ni exigir la ficha de procedencia. Eso permitía
    que A2 y A3 recortaran con límites distintos sin que nada lo advirtiera: si
    alguien cambiaba el límite en un cuaderno y no en el otro, el Bloque B
    terminaría emparejando dos inventarios definidos sobre áreas diferentes, y
    las estaciones que solo aparecieran en una fuente podrían deberse a eso y
    no a una diferencia real de cobertura. Ahora ambos cuadernos llaman a la
    misma función, con la misma configuración, declarada en un solo lugar.

    POR QUÉ SE FILTRA POR CÓDIGO Y NO POR NOMBRE
    La capa del IGAC es nacional: trae los 1.105 municipios del país. Hay que
    quedarse con uno. Se selecciona por código DANE —11001 para Bogotá, 05001
    para Medellín— y no por nombre, porque en esta capa Bogotá figura como
    "Bogotá, D.C.", con coma y abreviatura, de modo que cualquier búsqueda por
    texto es frágil. El código es un identificador estable.

    columna_codigo y codigo son obligatorios aquí. Si se omiten, la función
    lanza excepción en lugar de disolver los 1.105 municipios en un solo
    polígono que abarcaría el país entero: un error que no daría ningún aviso
    y que dejaría pasar todas las estaciones nacionales como si fueran de la
    ciudad.

    Returns
    -------
    tuple(dict, dict)
        Límites disueltos por ciudad y sus fichas de procedencia, que se
        propagan a los metadatos del entregable.
    """

    limites = {}

    fichas = {}

    for ciudad, configuracion in AREAS_ESTUDIO.items():

        limite, ficha = cargar_limite_ciudad(
            archivo=configuracion["limite"]["archivo"],
            capa=configuracion["limite"]["capa"],
            nombre_ciudad=configuracion["nombre"],
            procedencia=configuracion["limite"]["procedencia"],
            columna_codigo=configuracion["limite"]["columna_codigo"],
            codigo=configuracion["limite"]["codigos_dane"][ciudad],
        )

        limites[ciudad] = limite

        fichas[ciudad] = ficha

        # --------------------------------------------------------------------
        # Verificación independiente del tamaño
        # --------------------------------------------------------------------
        # Se calcula el área en kilómetros cuadrados como comprobación de que
        # el filtro por código aisló un municipio y no algo más grande. Si el
        # filtro fallara y se disolviera la capa completa, el área saltaría a
        # cientos de miles de kilómetros cuadrados en lugar de cientos. Es una
        # magnitud que se puede contrastar con conocimiento externo, a
        # diferencia del conteo de filas, que sería 1 en ambos casos.

        area_km2 = (
            limite.to_crs(CRS_METRICO).area.iloc[0] / 1e6
        )

        if not 100 < area_km2 < 5000:

            raise AssertionError(
                f"{ficha['ciudad']}: el límite cargado tiene {area_km2:,.1f} "
                "km², fuera del rango esperable para un municipio. Es "
                "probable que el filtro por código no haya funcionado y se "
                "haya disuelto más de un registro."
            )

        print(
            f"{ficha['ciudad']}: código {ficha['codigo_filtrado']} | "
            f"{ficha['registros_en_la_capa']} registros en la capa | "
            f"CRS origen {ficha['crs_original']} | "
            f"área {area_km2:,.1f} km²"
        )

    return limites, fichas

# Filtrado espacial

In [10]:
def filtrar_ocm_por_ciudad(
    gdf_colombia,
    limites,
    df_conexiones,
):
    """
    Recorta el inventario nacional a cada ciudad de estudio.

    QUÉ PRODUCE
    Tres cosas por ciudad: las estaciones que caen dentro del límite, sus
    conexiones correspondientes, y la auditoría del recorte.

    POR QUÉ SE AUDITA LO QUE QUEDA FUERA
    La versión anterior se quedaba solo con lo que entraba. Las estaciones
    descartadas desaparecían sin dejar rastro, y eso hace imposible responder
    una pregunta que un revisor sí va a hacer: ¿el límite descartó lo que
    debía? En A2 el mismo recorte dejó fuera estaciones de Cota, Tenjo,
    Envigado e Itagüí, y comprobar que esos nombres correspondieran a
    municipios vecinos fue lo que dio confianza en el filtro. Aquí se hace lo
    mismo: se conservan las excluidas con el municipio que declara la fuente,
    para poder contrastar la asignación del IGAC contra lo que dice
    OpenChargeMap.

    POR QUÉ SE MIDE LA DISTANCIA AL BORDE
    La capa del IGAC declara que no es apropiada para la ubicación exacta de
    puntos. Esa restricción solo importa si alguna estación queda tan cerca
    del límite que un error de trazado pudiera cambiar su asignación. Se mide
    la distancia de cada estación al borde y se cuentan las que están dentro
    de una franja de tolerancia. Si ninguna lo está, la restricción declarada
    por la fuente no afecta a este inventario, y eso se puede afirmar con un
    número en lugar de suponerlo.

    Returns
    -------
    tuple(dict, dict, dict)
        Inventarios por ciudad, conexiones por ciudad y auditoría del recorte.
    """

    # Franja alrededor del límite dentro de la cual la asignación de una
    # estación a un municipio podría verse afectada por el trazado de la
    # frontera. 50 m es holgado frente a la precisión de una capa 1:100.000
    # y frente al error típico de un GPS urbano.
    TOLERANCIA_FRONTERA_M = 50.0

    inventarios = {}

    conexiones_ciudades = {}

    auditoria = {}

    for ciudad, limite in limites.items():

        # --------------------------------------------------------------------
        # Recorte
        # --------------------------------------------------------------------
        # Se usa sjoin con predicado 'within' y no 'intersects': una estación
        # es un punto, de modo que ambos coinciden salvo en el borde exacto,
        # pero 'within' expresa lo que se quiere decir.

        dentro = gpd.sjoin(
            gdf_colombia,
            limite,
            how="inner",
            predicate="within",
        ).drop(columns="index_right")

        # El sjoin puede duplicar filas si el límite tuviera varias
        # geometrías. Se disolvió antes, de modo que no debería ocurrir;
        # comprobarlo es barato y el error sería difícil de rastrear después.

        if dentro["ocm_id"].duplicated().any():

            raise AssertionError(
                f"{ciudad}: el recorte duplicó estaciones. El límite no "
                "quedó disuelto en una sola geometría."
            )

        dentro = dentro.reset_index(drop=True)

        # --------------------------------------------------------------------
        # Excluidas
        # --------------------------------------------------------------------

        ids_dentro = set(dentro["ocm_id"])

        fuera = gdf_colombia[
            ~gdf_colombia["ocm_id"].isin(ids_dentro)
        ].copy()

        # --------------------------------------------------------------------
        # Balance del recorte
        # --------------------------------------------------------------------

        if len(dentro) + len(fuera) != len(gdf_colombia):

            raise AssertionError(
                f"{ciudad}: el recorte no cuadra. {len(gdf_colombia)} "
                f"estaciones nacionales, {len(dentro)} dentro y "
                f"{len(fuera)} fuera."
            )

        # --------------------------------------------------------------------
        # Distancia al borde
        # --------------------------------------------------------------------
        # En metros, sobre el CRS métrico oficial. La distancia se mide contra
        # el contorno del polígono, no contra el polígono: la distancia de un
        # punto interior a un polígono que lo contiene es cero.

        borde = limite.to_crs(CRS_METRICO).geometry.iloc[0].boundary

        if len(dentro):

            distancias = (
                dentro.to_crs(CRS_METRICO)
                .geometry
                .distance(borde)
            )

            en_frontera = int(
                (distancias < TOLERANCIA_FRONTERA_M).sum()
            )

            distancia_minima = float(distancias.min())

        else:

            en_frontera = 0

            distancia_minima = None

        # --------------------------------------------------------------------
        # Conexiones de las estaciones que quedaron dentro
        # --------------------------------------------------------------------

        conexiones_ciudad = df_conexiones[
            df_conexiones["ocm_id"].isin(ids_dentro)
        ].copy().reset_index(drop=True)

        inventarios[ciudad] = dentro

        conexiones_ciudades[ciudad] = conexiones_ciudad

        # --------------------------------------------------------------------
        # Municipios declarados por la fuente entre las excluidas
        # --------------------------------------------------------------------
        # Sirve para contrastar la asignación del IGAC contra lo que dice
        # OpenChargeMap. No son fuentes independientes en sentido estricto,
        # pero una discrepancia sistemática sería señal de problema.

        municipios_excluidos = (
            fuera["ciudad"]
            .fillna("(sin municipio declarado)")
            .value_counts()
            .head(10)
            .to_dict()
        )

        auditoria[ciudad] = {
            "estaciones_nacionales": len(gdf_colombia),
            "dentro_del_limite": len(dentro),
            "fuera_del_limite": len(fuera),
            "conexiones_dentro": len(conexiones_ciudad),
            "estaciones_en_franja_frontera": en_frontera,
            "tolerancia_frontera_m": TOLERANCIA_FRONTERA_M,
            "distancia_minima_al_borde_m": distancia_minima,
            "municipios_declarados_entre_excluidas": municipios_excluidos,
        }

        # --------------------------------------------------------------------
        # Reporte
        # --------------------------------------------------------------------

        print(f"\n{ciudad}")

        print(
            f"  Dentro del límite: {len(dentro)} de "
            f"{len(gdf_colombia)} estaciones nacionales"
        )

        print(
            f"  Conexiones asociadas: {len(conexiones_ciudad)}"
        )

        if distancia_minima is not None:

            print(
                f"  Distancia mínima al borde: "
                f"{distancia_minima:,.1f} m"
            )

            if en_frontera:

                print(
                    f"  ⚠ {en_frontera} estaciones a menos de "
                    f"{TOLERANCIA_FRONTERA_M:.0f} m del límite: su asignación "
                    "depende del trazado y debe declararse."
                )

            else:

                print(
                    f"  Ninguna estación a menos de "
                    f"{TOLERANCIA_FRONTERA_M:.0f} m del límite: la "
                    "restricción de exactitud declarada por el IGAC no "
                    "afecta a este inventario."
                )

        municipios_cercanos = {
            municipio: conteo
            for municipio, conteo in municipios_excluidos.items()
            if municipio != "(sin municipio declarado)"
        }

        if municipios_cercanos:

            muestra = ", ".join(
                f"{municipio} ({conteo})"
                for municipio, conteo in list(
                    municipios_cercanos.items()
                )[:5]
            )

            print(f"  Municipios entre las excluidas: {muestra}")

    return inventarios, conexiones_ciudades, auditoria

# Análisis de completitud

In [11]:
# ============================================================================
# Valores que ocupan la celda pero no informan nada
# ============================================================================
# Contar solo los nulos daría una completitud inflada. OpenChargeMap usa
# cadenas literales como marcador de ausencia, de modo que la celda está
# llena pero el dato no existe. Se listan aquí para que el criterio quede
# explícito y no escondido dentro de la función.

CENTINELAS_OCM = {

    # Literal que la fuente escribe cuando no conoce al operador. No es un
    # operador llamado así.
    "operador": ["(Unknown Operator)"],

    # 'Application' significa que la tarifa depende de la aplicación del
    # proveedor, es decir, que no está declarada aquí.
    "costo_uso": ["Application"],
}


def calcular_completitud(
    df_estaciones,
    df_conexiones=None,
    etiqueta="",
):
    """
    Mide qué proporción de cada campo viene efectivamente diligenciada.

    QUÉ RESPONDE
    La revisión del Bloque A pidió el conteo de campos vacíos porque, para el
    ruteo, un inventario con coordenadas pero sin potencia ni conector es
    inservible: la coordenada dice dónde está la estación, no si el vehículo
    puede cargar en ella.

    POR QUÉ SE SEPARAN NULOS DE CENTINELAS
    Un campo puede estar lleno y no informar nada. 'operador' con valor
    "(Unknown Operator)" no es un operador: es la manera que tiene la fuente
    de decir que no sabe. Contarlo como presente inflaría la completitud
    justo en los campos donde más importa no engañarse. Se reportan las dos
    cifras por separado para que la diferencia quede a la vista.

    POR QUÉ LA POTENCIA SE MIDE APARTE
    La potencia vive en la tabla de conexiones, no en la de estaciones. Una
    estación puede tener tres conectores y declarar la potencia de uno solo.
    Medirla sobre las estaciones ocultaría eso.

    Returns
    -------
    pandas.DataFrame
        Una fila por campo, ordenada de menor a mayor completitud.
    """

    total = len(df_estaciones)

    filas = []

    for columna in df_estaciones.columns:

        if columna == "geometry":
            continue

        serie = df_estaciones[columna]

        nulos = int(serie.isna().sum())

        centinelas = 0

        valores_centinela = CENTINELAS_OCM.get(columna, [])

        if valores_centinela:

            centinelas = int(
                serie.astype("string")
                .isin(valores_centinela)
                .sum()
            )

        # Las cadenas vacías o de solo espacios cuentan como ausencia.

        if serie.dtype == object:

            vacias = int(
                serie.astype("string")
                .str.strip()
                .eq("")
                .fillna(False)
                .sum()
            )

        else:

            vacias = 0

        sin_dato = nulos + centinelas + vacias

        filas.append({

            "campo": columna,

            "total": total,

            "con_dato": total - sin_dato,

            "sin_dato": sin_dato,

            "nulos": nulos,

            "centinelas": centinelas,

            "cadenas_vacias": vacias,

            "completitud_pct": (
                round(100 * (total - sin_dato) / total, 1)
                if total
                else 0.0
            ),

            "centinelas_considerados": (
                " | ".join(valores_centinela)
                if valores_centinela
                else ""
            ),
        })

    tabla = (
        pd.DataFrame(filas)
        .sort_values("completitud_pct")
        .reset_index(drop=True)
    )

    # ------------------------------------------------------------------------
    # Potencia, medida sobre las conexiones
    # ------------------------------------------------------------------------

    resumen_potencia = None

    if df_conexiones is not None and not df_conexiones.empty:

        conexiones_ciudad = df_conexiones[
            df_conexiones["ocm_id"].isin(df_estaciones["ocm_id"])
        ]

        if len(conexiones_ciudad):

            con_potencia = int(
                conexiones_ciudad["potencia_kw"].notna().sum()
            )

            estaciones_con_potencia = int(
                conexiones_ciudad
                .loc[
                    conexiones_ciudad["potencia_kw"].notna(),
                    "ocm_id",
                ]
                .nunique()
            )

            resumen_potencia = {
                "conexiones": len(conexiones_ciudad),
                "conexiones_con_potencia": con_potencia,
                "estaciones_con_alguna_potencia": (
                    estaciones_con_potencia
                ),
                "estaciones": total,
            }

    # ------------------------------------------------------------------------
    # Reporte
    # ------------------------------------------------------------------------

    print(f"\nCompletitud{f' — {etiqueta}' if etiqueta else ''}")

    print(f"  Estaciones evaluadas: {total}")

    criticos = tabla[
        tabla["campo"].isin([
            "nombre",
            "operador",
            "latitud",
            "longitud",
            "tipo_uso",
            "costo_uso",
            "estado_operativo",
            "fecha_ultima_verificacion",
        ])
    ]

    print("\n  Campos relevantes para el ruteo:")

    for fila in criticos.itertuples():

        nota = ""

        if fila.centinelas:
            nota = (
                f"  ({fila.centinelas} con valor centinela, "
                "contados como ausencia)"
            )

        print(
            f"    {fila.campo:28} {fila.completitud_pct:5.1f}%  "
            f"({fila.con_dato}/{fila.total}){nota}"
        )

    if resumen_potencia:

        print("\n  Potencia (medida sobre conexiones):")

        print(
            f"    Conexiones con potencia declarada: "
            f"{resumen_potencia['conexiones_con_potencia']}"
            f"/{resumen_potencia['conexiones']}"
        )

        print(
            f"    Estaciones con al menos una potencia: "
            f"{resumen_potencia['estaciones_con_alguna_potencia']}"
            f"/{resumen_potencia['estaciones']}"
        )

    vacios_del_todo = tabla[tabla["completitud_pct"] == 0.0]

    if len(vacios_del_todo):

        nombres = ", ".join(vacios_del_todo["campo"].head(8))

        print(
            f"\n  ⚠ Campos sin ningún dato ({len(vacios_del_todo)}): "
            f"{nombres}"
        )

        print(
            "    Un campo vacío en todos los registros no discrimina nada "
            "y no debe usarse como criterio."
        )

    return tabla

# Validación por ciudad

In [12]:
def validar_inventarios_ciudades(
    inventarios_ciudades,
    conexiones_ciudades,
):
    """
    Ejecuta la validación estructural sobre cada inventario urbano.

    La versión anterior llamaba a validar_inventario_ocm sin la tabla de
    conexiones y metía el resultado —que ya es un DataFrame— dentro de otro
    DataFrame, lo que producía una tabla anidada ilegible. Aquí se concatenan
    las comprobaciones de cada ciudad en una sola tabla plana.
    """

    resultados = []

    for ciudad, gdf in inventarios_ciudades.items():

        validacion = validar_inventario_ocm(
            gdf.drop(columns="geometry"),
            conexiones_ciudades[ciudad],
        )

        validacion.insert(0, "ciudad", ciudad)

        resultados.append(validacion)

        # Comprobaciones que solo aplican a la capa geográfica.

        extra = pd.DataFrame([
            {
                "ciudad": ciudad,
                "comprobacion": "Geometrías no vacías",
                "valor": int((~gdf.geometry.is_empty).sum()),
                "detalle": f"de {len(gdf)} estaciones",
            },
            {
                "ciudad": ciudad,
                "comprobacion": "CRS de la capa",
                "valor": len(gdf),
                "detalle": gdf.crs.to_string(),
            },
        ])

        if gdf.geometry.is_empty.any():

            raise AssertionError(
                f"{ciudad}: hay geometrías vacías en el inventario urbano."
            )

        resultados.append(extra)

    return pd.concat(resultados, ignore_index=True)

# Exportación

In [13]:
def exportar_resultados_a3(
    df_colombia,
    df_conexiones_colombia,
    inventarios_ciudades,
    conexiones_ciudades,
    completitud_colombia,
    completitud_ciudades,
    resultado_descarga,
    fichas_limites,
    analisis_vigencia,
    auditoria_transformacion,
    auditoria_recorte,
    estaciones_sin_coordenadas,
):
    """
    Exporta los productos del entregable A3.

    QUÉ CAMBIÓ Y POR QUÉ

    1. Ya no se escribe el archivo crudo. Antes esta función reserializaba
       los datos en A3_OpenChargeMap_Colombia_crudo.json. Ahora el crudo es
       un INSUMO congelado que vive en la carpeta con su hash y su ficha de
       metadatos, y que el cuaderno lee sin tocar la red. Volver a escribirlo
       produciría un segundo archivo con el mismo contenido pero sin huella
       verificable, y no habría forma de saber cuál de los dos es el bueno.

    2. Todas las salidas se escriben con ruta_entregable("A3", ...). Antes
       usaban nombres relativos sueltos, de modo que los archivos caían en la
       carpeta desde la que se hubiera lanzado el kernel, que no siempre es
       la del entregable.

    3. Se exportan las auditorías y las estaciones sin coordenadas. No son
       subproductos del procesamiento: son los conteos que sostienen las
       pruebas de aceptación, y deben quedar en disco para que otra persona
       pueda revisarlos sin reejecutar.
    """

    archivos = {}

    def guardar_csv(tabla, nombre, clave):

        ruta = ruta_entregable("A3", nombre)

        ruta.parent.mkdir(parents=True, exist_ok=True)

        tabla.to_csv(ruta, index=False, encoding="utf-8-sig")

        archivos[clave] = str(ruta)

        return ruta

    # ------------------------------------------------------------------------
    # Tablas nacionales
    # ------------------------------------------------------------------------

    guardar_csv(
        df_colombia,
        "A3_OpenChargeMap_Colombia_estaciones.csv",
        "estaciones_colombia",
    )

    guardar_csv(
        df_conexiones_colombia,
        "A3_OpenChargeMap_Colombia_conexiones.csv",
        "conexiones_colombia",
    )

    # ------------------------------------------------------------------------
    # Tablas por ciudad
    # ------------------------------------------------------------------------

    for ciudad, gdf in inventarios_ciudades.items():

        guardar_csv(
            gdf.drop(columns="geometry"),
            f"A3_OpenChargeMap_{ciudad}_estaciones.csv",
            f"estaciones_{ciudad}",
        )

        guardar_csv(
            conexiones_ciudades[ciudad],
            f"A3_OpenChargeMap_{ciudad}_conexiones.csv",
            f"conexiones_{ciudad}",
        )

        # Capa geográfica, para poder abrir el inventario en un SIG sin
        # reconstruir la geometría desde las columnas de coordenadas.

        ruta_geojson = ruta_entregable(
            "A3", f"A3_OpenChargeMap_{ciudad}_estaciones.geojson"
        )

        gdf.to_file(ruta_geojson, driver="GeoJSON")

        archivos[f"geojson_{ciudad}"] = str(ruta_geojson)

    # ------------------------------------------------------------------------
    # Completitud
    # ------------------------------------------------------------------------

    guardar_csv(
        completitud_colombia,
        "A3_OpenChargeMap_Colombia_completitud.csv",
        "completitud_colombia",
    )

    for ciudad, tabla in completitud_ciudades.items():

        guardar_csv(
            tabla,
            f"A3_OpenChargeMap_{ciudad}_completitud.csv",
            f"completitud_{ciudad}",
        )

    # ------------------------------------------------------------------------
    # Vigencia
    # ------------------------------------------------------------------------

    guardar_csv(
        analisis_vigencia,
        "A3_OpenChargeMap_vigencia.csv",
        "vigencia",
    )

    # ------------------------------------------------------------------------
    # Estaciones sin coordenadas
    # ------------------------------------------------------------------------
    # Se exportan aunque estén vacías: la ausencia del archivo sería
    # ambigua —no se sabría si no hubo ninguna o si no se revisó—, mientras
    # que un archivo con cero filas afirma que se comprobó.

    guardar_csv(
        estaciones_sin_coordenadas,
        "A3_OpenChargeMap_sin_coordenadas.csv",
        "sin_coordenadas",
    )

    # ------------------------------------------------------------------------
    # Auditorías
    # ------------------------------------------------------------------------

    ruta_auditoria = ruta_entregable(
        "A3", "A3_auditoria_procesamiento.json"
    )

    auditoria = {
        "corte": {
            "fecha_corte_local": resultado_descarga["fecha_corte_local"],
            "fecha_descarga_declarada": resultado_descarga[
                "fecha_descarga_declarada"
            ],
            "origen": resultado_descarga["origen"],
            "ruta_crudo": resultado_descarga["ruta_crudo"],
            "hash_crudo": resultado_descarga["hash_crudo"],
            "fechas_maximas_declaradas": resultado_descarga[
                "fechas_maximas_declaradas"
            ],
        },
        "transformacion": auditoria_transformacion,
        "recorte_espacial": auditoria_recorte,
        "limites": {
            ciudad: {
                clave: str(valor)
                for clave, valor in ficha.items()
            }
            for ciudad, ficha in fichas_limites.items()
        },
    }

    with open(ruta_auditoria, "w", encoding="utf-8") as archivo:
        json.dump(auditoria, archivo, ensure_ascii=False, indent=2)

    archivos["auditoria"] = str(ruta_auditoria)

    # ------------------------------------------------------------------------
    # Metadatos legibles
    # ------------------------------------------------------------------------

    fichas_texto = ""

    for ciudad, ficha in fichas_limites.items():

        fichas_texto += (
            f"\n{ciudad}:\n"
            f"  archivo: {ficha['archivo']}\n"
            f"  capa: {ficha['capa']}\n"
            f"  código filtrado: {ficha['codigo_filtrado']}\n"
            f"  entidad: {ficha['entidad']}\n"
            f"  url: {ficha['url']}\n"
            f"  fecha de descarga: {ficha['fecha_descarga_limite']}\n"
            f"  CRS de origen: {ficha['crs_original']}\n"
        )

    metadata = f"""
ENTREGABLE A3 — OPENCHARGEMAP
=============================

Fuente: OpenChargeMap POI API v3
País consultado: Colombia

LICENCIA Y ATRIBUCIÓN
---------------------
{ATRIBUCION_OCM}

CORTE TEMPORAL
--------------
Estado representado:      {resultado_descarga["fecha_corte_local"]}
Fecha de descarga:        {resultado_descarga["fecha_descarga_declarada"]}
Origen de los datos:      {resultado_descarga["origen"]}

Archivo crudo congelado:
{resultado_descarga["ruta_crudo"]}

SHA-256:
{resultado_descarga["hash_crudo"]}

OpenChargeMap no admite consultas de estados históricos, a diferencia de
Overpass en A2. El archivo crudo ES el corte: si se pierde, el inventario
no puede reconstruirse, porque cualquier consulta posterior devolverá el
estado del día en que se haga.

La descarga es posterior al estado que se le atribuye. Lo que sostiene esa
atribución es que ninguna de las fechas que declara la fuente supera el
corte:

{json.dumps(resultado_descarga["fechas_maximas_declaradas"], indent=2, ensure_ascii=False)}

LÍMITE DE ESA VERIFICACIÓN: si OpenChargeMap modificó algún registro entre
el corte y la descarga sin actualizar ninguno de esos campos, no hay forma
de detectarlo con los datos que la fuente publica.

PARÁMETROS DE CONSULTA
----------------------
{json.dumps(resultado_descarga["parametros"], indent=2, ensure_ascii=False)}

Registros recuperados: {resultado_descarga["n_registros"]}
Límite solicitado: {OPENCHARGEMAP_MAX_RESULTS}
¿Posible truncamiento?: {"SÍ — revisar" if resultado_descarga["truncado"] else "no"}

BALANCE DE LA TRANSFORMACIÓN
----------------------------
{json.dumps(auditoria_transformacion, indent=2, ensure_ascii=False)}

RECORTE ESPACIAL
----------------
{json.dumps(auditoria_recorte, indent=2, ensure_ascii=False)}

LÍMITES EMPLEADOS
-----------------{fichas_texto}
INVENTARIOS
-----------
Estaciones (Colombia): {len(df_colombia)}
Conexiones (Colombia): {len(df_conexiones_colombia)}
"""

    for ciudad, gdf in inventarios_ciudades.items():

        metadata += (
            f"\nEstaciones {ciudad}: {len(gdf)}"
            f"\nConexiones {ciudad}: "
            f"{len(conexiones_ciudades[ciudad])}\n"
        )

    metadata += "\nARCHIVOS GENERADOS\n------------------\n"

    for clave, nombre in archivos.items():
        metadata += f"{clave}: {os.path.basename(nombre)}\n"

    ruta_metadata = ruta_entregable(
        "A3", "A3_OpenChargeMap_metadata_descarga.txt"
    )

    with open(ruta_metadata, "w", encoding="utf-8") as archivo:
        archivo.write(metadata.strip())

    archivos["metadata"] = str(ruta_metadata)

    print(
        f"\n✓ {len(archivos)} productos exportados a "
        f"{ruta_entregable('A3')}"
    )

    return archivos

# Ejecución completa de A3

In [14]:
# ============================================================================
# 14.1. DESCARGA — lectura del crudo congelado
# ============================================================================

resultado_descarga_a3 = descargar_ocm_colombia()

datos_ocm_a3 = resultado_descarga_a3["datos"]

# La fecha que acompaña a cada registro es la de la DESCARGA declarada, no la
# de finalización del proceso: cuando se lee del archivo congelado no hubo
# proceso, y 'fecha_fin' vale None.

fecha_descarga_ocm_a3 = resultado_descarga_a3[
    "fecha_descarga_declarada"
]

# ============================================================================
# 14.2. TRANSFORMACIÓN
# ============================================================================

(
    df_ocm_colombia,
    df_conexiones_colombia,
    auditoria_transformacion_a3,
) = transformar_registros_ocm(
    datos_ocm_a3,
    fecha_descarga_ocm_a3,
)

# ============================================================================
# 14.3. VALIDACIÓN NACIONAL
# ============================================================================

validacion_colombia_a3 = validar_inventario_ocm(
    df_ocm_colombia,
    df_conexiones_colombia,
    auditoria=auditoria_transformacion_a3,
)

print("\nValidación nacional:")

display(validacion_colombia_a3)

# ============================================================================
# 14.4. CONSTRUCCIÓN GEOGRÁFICA
# ============================================================================

(
    gdf_ocm_colombia,
    estaciones_sin_coordenadas_a3,
) = convertir_a_geodataframe(df_ocm_colombia)

# ============================================================================
# 14.5. CARGA DE LÍMITES
# ============================================================================

(
    limites_estudio_a3,
    fichas_limites_a3,
) = cargar_limites_estudio()

# ============================================================================
# 14.6. FILTRADO POR CIUDAD
# ============================================================================

(
    inventarios_ocm_ciudades,
    conexiones_ocm_ciudades,
    auditoria_recorte_a3,
) = filtrar_ocm_por_ciudad(
    gdf_ocm_colombia,
    limites_estudio_a3,
    df_conexiones_colombia,
)

# ============================================================================
# 14.7. COMPLETITUD NACIONAL
# ============================================================================

completitud_ocm_colombia = calcular_completitud(
    df_ocm_colombia,
    df_conexiones_colombia,
    etiqueta="Colombia",
)

# ============================================================================
# 14.8. COMPLETITUD POR CIUDAD
# ============================================================================

completitud_ocm_ciudades = {
    ciudad: calcular_completitud(
        gdf.drop(columns="geometry"),
        conexiones_ocm_ciudades[ciudad],
        etiqueta=ciudad,
    )
    for ciudad, gdf in inventarios_ocm_ciudades.items()
}

# ============================================================================
# 14.9. VALIDACIÓN POR CIUDAD
# ============================================================================

validacion_ciudades_a3 = validar_inventarios_ciudades(
    inventarios_ocm_ciudades,
    conexiones_ocm_ciudades,
)

display(validacion_ciudades_a3)

# ============================================================================
# 14.10. ANÁLISIS DE VIGENCIA
# ============================================================================


def analizar_vigencia(df, nombre, fecha_corte_local=FECHA_CORTE_LOCAL):
    """
    Distribución de la antigüedad de la última verificación.

    La revisión señaló que este campo se descargó y no se usó, y que para un
    modelo que decide si una ruta es factible, una estación sin verificar
    desde hace años no puede tratarse igual que una verificada este año.

    POR QUÉ SE MIDE CONTRA EL CORTE Y NO CONTRA HOY
    La versión anterior calculaba la antigüedad contra la fecha de ejecución.
    Eso hacía que los conteos cambiaran solos con el paso del tiempo: una
    estación que hoy cae en "≤ 1 año" pasaría a "1 a 2 años" dentro de unos
    meses sin que el dato hubiera cambiado, y dos ejecuciones del mismo
    cuaderno sobre el mismo archivo darían resultados distintos. Medir contra
    el corte fija el resultado: el inventario describe un instante, y la
    antigüedad se mide desde ese instante.
    """

    referencia = pd.Timestamp(
        f"{fecha_corte_local}T23:59:59",
        tz="America/Bogota",
    ).tz_convert("UTC")

    fechas = pd.to_datetime(
        df["fecha_ultima_verificacion"],
        errors="coerce",
        utc=True,
    )

    antiguedad_dias = (referencia - fechas).dt.days

    filas = []

    for etiqueta, mascara in [
        ("sin fecha de verificación", fechas.isna()),
        ("verificada ≤ 1 año", antiguedad_dias <= 365),
        (
            "entre 1 y 2 años",
            (antiguedad_dias > 365) & (antiguedad_dias <= 730),
        ),
        ("más de 2 años", antiguedad_dias > 730),
    ]:

        filas.append({
            "inventario": nombre,
            "categoria": etiqueta,
            "registros": int(mascara.sum()),
            "porcentaje": (
                round(mascara.sum() / len(df) * 100, 2)
                if len(df)
                else 0.0
            ),
            "referencia": fecha_corte_local,
        })

    tabla = pd.DataFrame(filas)

    # Verificación independiente: las cuatro categorías son excluyentes y
    # deben cubrir todos los registros.

    if tabla["registros"].sum() != len(df):

        raise AssertionError(
            f"{nombre}: las categorías de vigencia suman "
            f"{tabla['registros'].sum()} y hay {len(df)} registros."
        )

    return tabla


vigencia_a3 = pd.concat(
    [analizar_vigencia(df_ocm_colombia, "Colombia")]
    + [
        analizar_vigencia(gdf.drop(columns="geometry"), ciudad)
        for ciudad, gdf in inventarios_ocm_ciudades.items()
    ],
    ignore_index=True,
)

display(vigencia_a3)

# ============================================================================
# 14.11. EXPORTACIÓN
# ============================================================================

archivos_a3 = exportar_resultados_a3(
    df_colombia=df_ocm_colombia,
    df_conexiones_colombia=df_conexiones_colombia,
    inventarios_ciudades=inventarios_ocm_ciudades,
    conexiones_ciudades=conexiones_ocm_ciudades,
    completitud_colombia=completitud_ocm_colombia,
    completitud_ciudades=completitud_ocm_ciudades,
    resultado_descarga=resultado_descarga_a3,
    fichas_limites=fichas_limites_a3,
    analisis_vigencia=vigencia_a3,
    auditoria_transformacion=auditoria_transformacion_a3,
    auditoria_recorte=auditoria_recorte_a3,
    estaciones_sin_coordenadas=estaciones_sin_coordenadas_a3,
)


A3 — LECTURA DEL ARCHIVO CRUDO CONGELADO

Archivo: F:\Tesis\Bloque A\Entregable A3\A3_ocm_crudo_Colombia_estado-al-2026-08-29_bajado-2026-09-02.json
  SHA-256 calculado y registrado por primera vez: c281a19abe3fdc10
  Corte compatible: ninguna de las 4 fechas declaradas supera el 2026-08-29.
    DateCreated: máximo 2026-08-21T16:53:00+00:00
    DateLastStatusUpdate: máximo 2026-08-24T16:56:00+00:00
    DateLastVerified: máximo 2026-08-24T16:56:00+00:00
    DateLastConfirmed: máximo None
  Metadatos generados: F:\Tesis\Bloque A\Entregable A3\A3_ocm_crudo_Colombia_estado-al-2026-08-29_bajado-2026-09-02.meta.json
  119 registros leídos sin consultar la API.

✓ Estaciones: 119 filas × 40 columnas
✓ Conexiones: 281 filas × 18 columnas

  Balance: 119 recibidos = 119 transformados + 0 sin identificador
  Estaciones sin coordenadas utilizables: 0
  Estaciones sin ninguna conexión declarada: 0

Validación nacional:


,comprobacion,valor,detalle
0,Estaciones,119,40 columnas
1,Conexiones,281,18 columnas
2,ocm_id únicos,119,sin duplicados
3,Integridad referencial,119,todo ocm_id de conexiones existe en estaciones
4,Verificación independiente: potencia máxima,114,"recalculada desde conexiones, coincide"
5,Estaciones sin coordenadas,0,se conservan en la tabla; no entrarán a ningun...
6,Coordenadas dentro del territorio,119,"lat [-4.5, 13.5], lon [-82.0, -66.0]"
7,Balance de la transformación,119,119 = 119 transformados + 0 sin identificador
8,Sin coordenadas: recorrido vs tabla,0,ambos caminos coinciden



✓ Capa de puntos: 119 estaciones en EPSG:4326
  Geometría verificada contra las columnas latitud/longitud.
Bogotá D.C.: código 11001 | 1105 registros en la capa | CRS origen EPSG:9377 | área 1,618.2 km²
Medellín: código 05001 | 1105 registros en la capa | CRS origen EPSG:9377 | área 373.4 km²

Bogota
  Dentro del límite: 27 de 119 estaciones nacionales
  Conexiones asociadas: 56
  Distancia mínima al borde: 1,763.2 m
  Ninguna estación a menos de 50 m del límite: la restricción de exactitud declarada por el IGAC no afecta a este inventario.
  Municipios entre las excluidas: Pereira (6), Medellín (6), Medellin (4), Puerto Triunfo (3), El Carmen de Viboral (3)

Medellin
  Dentro del límite: 11 de 119 estaciones nacionales
  Conexiones asociadas: 19
  Distancia mínima al borde: 62.2 m
  Ninguna estación a menos de 50 m del límite: la restricción de exactitud declarada por el IGAC no afecta a este inventario.
  Municipios entre las excluidas: Bogota (14), Pereira (6), Puerto Triunfo (3), 

,ciudad,comprobacion,valor,detalle
0,Bogota,Estaciones,27,41 columnas
1,Bogota,Conexiones,56,18 columnas
2,Bogota,ocm_id únicos,27,sin duplicados
3,Bogota,Integridad referencial,27,todo ocm_id de conexiones existe en estaciones
4,Bogota,Verificación independiente: potencia máxima,23,"recalculada desde conexiones, coincide"
5,Bogota,Estaciones sin coordenadas,0,se conservan en la tabla; no entrarán a ningun...
6,Bogota,Coordenadas dentro del territorio,27,"lat [-4.5, 13.5], lon [-82.0, -66.0]"
7,Bogota,Geometrías no vacías,27,de 27 estaciones
8,Bogota,CRS de la capa,27,EPSG:4326
9,Medellin,Estaciones,11,41 columnas


,inventario,categoria,registros,porcentaje,referencia
0,Colombia,sin fecha de verificación,0,0.00,2026-08-29
1,Colombia,verificada ≤ 1 año,89,74.79,2026-08-29
2,Colombia,entre 1 y 2 años,12,10.08,2026-08-29
3,Colombia,más de 2 años,18,15.13,2026-08-29
4,Bogota,sin fecha de verificación,0,0.00,2026-08-29
5,Bogota,verificada ≤ 1 año,17,62.96,2026-08-29
6,Bogota,entre 1 y 2 años,2,7.41,2026-08-29
7,Bogota,más de 2 años,8,29.63,2026-08-29
8,Medellin,sin fecha de verificación,0,0.00,2026-08-29
9,Medellin,verificada ≤ 1 año,8,72.73,2026-08-29



✓ 15 productos exportados a F:\Tesis\Bloque A\Entregable A3


# Resumen y pruebas de aceptación

In [15]:
# 15.1. RESUMEN
# ============================================================================

print("=" * 78)
print("ENTREGABLE A3 — RESUMEN")
print("=" * 78)

print(f"\nEstado representado: {FECHA_CORTE_LOCAL}")
print(f"Descargado el:       {FECHA_DESCARGA_OCM}")
print(f"Origen:              {resultado_descarga_a3['origen']}")

resumen_a3 = pd.DataFrame([
    {
        "inventario": "Colombia",
        "estaciones": len(df_ocm_colombia),
        "conexiones": len(df_conexiones_colombia),
    }
] + [
    {
        "inventario": ciudad,
        "estaciones": len(gdf),
        "conexiones": len(conexiones_ocm_ciudades[ciudad]),
    }
    for ciudad, gdf in inventarios_ocm_ciudades.items()
])

display(resumen_a3)

# ============================================================================
# 15.2. PRUEBAS DE ACEPTACIÓN
# ============================================================================
# Cada prueba comprueba una afirmación por un camino distinto al que la
# produjo. Es la disciplina que dejó la revisión del Bloque A: el error del
# emparejamiento era invisible porque los números se veían razonables, y solo
# apareció al recalcularlos de otra manera.

pruebas_a3 = []


def prueba(nombre, condicion, detalle=""):

    pruebas_a3.append({
        "prueba": nombre,
        "resultado": "PASA" if condicion else "FALLA",
        "detalle": detalle,
    })


# ---------------------------------------------------------------------------
# Corte y procedencia
# ---------------------------------------------------------------------------

rutas_verificacion = rutas_crudo_ocm()

prueba(
    "1. El crudo congelado existe en disco",
    os.path.exists(rutas_verificacion["crudo"]),
    os.path.basename(rutas_verificacion["crudo"]),
)

hash_recalculado = sha256_archivo(rutas_verificacion["crudo"])

prueba(
    "2. Integridad del crudo: el hash coincide",
    hash_recalculado == resultado_descarga_a3["hash_crudo"],
    f"SHA-256 {hash_recalculado[:16]}",
)

prueba(
    "3. El cuaderno leyó del crudo y no de la API",
    resultado_descarga_a3["origen"] == "archivo crudo congelado",
    resultado_descarga_a3["origen"],
)

maximas = resultado_descarga_a3["fechas_maximas_declaradas"]

fechas_declaradas = [
    valor for valor in maximas.values() if valor
]

prueba(
    "4. Ninguna fecha declarada supera el corte",
    all(
        pd.Timestamp(valor)
        <= pd.Timestamp(
            f"{FECHA_CORTE_LOCAL}T23:59:59",
            tz="America/Bogota",
        ).tz_convert("UTC")
        for valor in fechas_declaradas
    ),
    f"máxima {max(fechas_declaradas) if fechas_declaradas else 'n/d'}",
)

prueba(
    "5. El inventario no está truncado",
    not resultado_descarga_a3["truncado"],
    (
        f"{resultado_descarga_a3['n_registros']} registros, "
        f"límite {OPENCHARGEMAP_MAX_RESULTS}"
    ),
)

# ---------------------------------------------------------------------------
# Balance de la transformación
# ---------------------------------------------------------------------------

aud = auditoria_transformacion_a3

prueba(
    "6. Balance: recibidos = transformados + descartados",
    (
        aud["registros_transformados"] + aud["registros_sin_id"]
        == aud["elementos_recibidos"]
    ),
    (
        f"{aud['elementos_recibidos']} = "
        f"{aud['registros_transformados']} + {aud['registros_sin_id']}"
    ),
)

prueba(
    "7. Los registros crudos y las filas de la tabla coinciden",
    len(datos_ocm_a3) == aud["elementos_recibidos"],
    f"{len(datos_ocm_a3)} elementos en el archivo",
)

# ---------------------------------------------------------------------------
# Estructura
# ---------------------------------------------------------------------------

prueba(
    "8. ocm_id único en el inventario nacional",
    df_ocm_colombia["ocm_id"].is_unique,
    f"{df_ocm_colombia['ocm_id'].nunique()} identificadores",
)

prueba(
    "9. Toda conexión pertenece a una estación existente",
    set(df_conexiones_colombia["ocm_id"]).issubset(
        set(df_ocm_colombia["ocm_id"])
    ),
    f"{len(df_conexiones_colombia)} conexiones",
)

# Recálculo independiente de la potencia máxima, desde las conexiones.

potencia_recalculada = (
    df_conexiones_colombia.groupby("ocm_id")["potencia_kw"].max()
)

potencia_declarada = (
    df_ocm_colombia.set_index("ocm_id")["potencia_kw_max"].dropna()
)

indices_comunes = potencia_declarada.index.intersection(
    potencia_recalculada.index
)

prueba(
    "10. Potencia máxima: recálculo independiente",
    bool(
        (
            potencia_declarada.loc[indices_comunes]
            == potencia_recalculada.loc[indices_comunes]
        ).all()
    ),
    f"{len(indices_comunes)} estaciones contrastadas",
)

# ---------------------------------------------------------------------------
# Recorte espacial
# ---------------------------------------------------------------------------

for ciudad, datos_recorte in auditoria_recorte_a3.items():

    prueba(
        f"11. Balance del recorte — {ciudad}",
        (
            datos_recorte["dentro_del_limite"]
            + datos_recorte["fuera_del_limite"]
            == datos_recorte["estaciones_nacionales"]
        ),
        (
            f"{datos_recorte['dentro_del_limite']} dentro + "
            f"{datos_recorte['fuera_del_limite']} fuera = "
            f"{datos_recorte['estaciones_nacionales']}"
        ),
    )

    prueba(
        f"12. Casos frontera — {ciudad}",
        datos_recorte["estaciones_en_franja_frontera"] == 0,
        (
            f"{datos_recorte['estaciones_en_franja_frontera']} estaciones a "
            f"menos de {datos_recorte['tolerancia_frontera_m']:.0f} m del "
            f"borde; mínima "
            f"{datos_recorte['distancia_minima_al_borde_m']:,.1f} m"
        ),
    )

# Cada estación urbana debe aparecer en una sola ciudad.

ids_por_ciudad = {
    ciudad: set(gdf["ocm_id"])
    for ciudad, gdf in inventarios_ocm_ciudades.items()
}

interseccion = set.intersection(*ids_por_ciudad.values())

prueba(
    "13. Ninguna estación pertenece a dos ciudades",
    len(interseccion) == 0,
    f"{len(interseccion)} estaciones compartidas",
)

# ---------------------------------------------------------------------------
# Salidas
# ---------------------------------------------------------------------------

faltantes = [
    clave
    for clave, ruta in archivos_a3.items()
    if not os.path.exists(ruta)
]

prueba(
    "14. Todos los archivos declarados existen",
    len(faltantes) == 0,
    f"{len(archivos_a3)} archivos; faltan {faltantes}",
)

prueba(
    "15. Las salidas quedaron en la carpeta del entregable",
    all(
        os.path.dirname(os.path.abspath(ruta))
        == str(ruta_entregable("A3"))
        for ruta in archivos_a3.values()
    ),
    str(ruta_entregable("A3")),
)

# El crudo no debe estar entre las salidas: es insumo, no producto.

prueba(
    "16. La exportación no reescribió el crudo",
    all(
        "ocm_crudo" not in os.path.basename(ruta)
        for ruta in archivos_a3.values()
    ),
    "el archivo congelado no figura entre los productos",
)

# ---------------------------------------------------------------------------
# Resultado
# ---------------------------------------------------------------------------

tabla_pruebas_a3 = pd.DataFrame(pruebas_a3)

display(tabla_pruebas_a3)

fallidas = tabla_pruebas_a3[
    tabla_pruebas_a3["resultado"] == "FALLA"
]

print(
    f"\n{len(tabla_pruebas_a3) - len(fallidas)}/"
    f"{len(tabla_pruebas_a3)} pruebas superadas."
)

ruta_pruebas = ruta_entregable("A3", "A3_pruebas_aceptacion.csv")

tabla_pruebas_a3.to_csv(
    ruta_pruebas, index=False, encoding="utf-8-sig"
)

if len(fallidas):

    raise AssertionError(
        f"{len(fallidas)} pruebas fallaron:\n"
        f"{fallidas.to_string(index=False)}"
    )

print("\n✓ Entregable A3 completo y verificado.")

ENTREGABLE A3 — RESUMEN

Estado representado: 2026-08-29
Descargado el:       2026-09-02
Origen:              archivo crudo congelado


,inventario,estaciones,conexiones
0,Colombia,119,281
1,Bogota,27,56
2,Medellin,11,19


,prueba,resultado,detalle
0,1. El crudo congelado existe en disco,PASA,A3_ocm_crudo_Colombia_estado-al-2026-08-29_baj...
1,2. Integridad del crudo: el hash coincide,PASA,SHA-256 c281a19abe3fdc10
2,3. El cuaderno leyó del crudo y no de la API,PASA,archivo crudo congelado
3,4. Ninguna fecha declarada supera el corte,PASA,máxima 2026-08-24T16:56:00+00:00
4,5. El inventario no está truncado,PASA,"119 registros, límite 5000"
5,6. Balance: recibidos = transformados + descar...,PASA,119 = 119 + 0
6,7. Los registros crudos y las filas de la tabl...,PASA,119 elementos en el archivo
7,8. ocm_id único en el inventario nacional,PASA,119 identificadores
8,9. Toda conexión pertenece a una estación exis...,PASA,281 conexiones
9,10. Potencia máxima: recálculo independiente,PASA,114 estaciones contrastadas



18/18 pruebas superadas.

✓ Entregable A3 completo y verificado.
